In [ ]:
import sys, platform, random, numpy as np, tensorflow as tf
print("Python:", sys.version)
print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

from google.colab import drive
drive.mount("/content/drive")
import os
PROJECT_DRIVE = "/content/drive/MyDrive/ocean_plastic"
os.makedirs(f"{PROJECT_DRIVE}/checkpoints", exist_ok=True)
# Test write/read on Drive
open(f"{PROJECT_DRIVE}/checkpoints/_test.txt","w").write("ok")
print("Drive read/write OK:", open(f"{PROJECT_DRIVE}/checkpoints/_test.txt").read())

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
REPO = "phanyen06ym-ien/ocean-plastic-ml-starter"

!git clone https://{token}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git config user.name "Hoang Bach"
!git config user.email "your_github_email@example.com"
!git checkout -b feature/s1-cnn-baseline
!mkdir -p src notebooks

In [ ]:
import tensorflow as tf
import numpy, matplotlib
with open("requirements.txt", "a") as f:   # append, so you don't overwrite teammates' lines
    f.write(f"\ntensorflow=={tf.__version__}\nnumpy=={numpy.__version__}\nmatplotlib=={matplotlib.__version__}\n")
!cat requirements.txt

In [ ]:
%%writefile src/train.py
import os, json, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DEFAULT_CONFIG = {
    "seed": 42,
    "input_shape": [128, 128, 3],   # TO CONFIRM with Hùng (S1-DATA)
    "num_classes": 4,               # PLACEHOLDER until Dataset Audit is done
    "filters": [32, 64, 128, 256],
    "dense_units": 128,
    "dropout": 0.5,
    "label_mode": "int",            # "int" -> sparse CE, "onehot" -> categorical CE
    "optimizer": "adam",
    "learning_rate": 1e-3,
    "batch_size": 32,
    "epochs": 30,
    "early_stopping_patience": 5,
    "normalization": "rescale_1/255",   # TO CONFIRM with Hùng
}

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def build_cnn(input_shape=(128, 128, 3), num_classes=4,
              filters=(32, 64, 128, 256), dense_units=128, dropout=0.5):
    """Baseline CNN: [Conv2D(3x3)+ReLU -> MaxPool2D] x N -> GAP -> Dense -> Softmax."""
    inputs = keras.Input(shape=tuple(input_shape), name="image")
    x = inputs
    for i, f in enumerate(filters):
        x = layers.Conv2D(f, 3, padding="same", activation="relu", name=f"conv{i+1}")(x)
        x = layers.MaxPooling2D(2, name=f"pool{i+1}")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(dense_units, activation="relu", name="fc1")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="cnn_baseline")

def compile_model(model, cfg):
    loss = ("sparse_categorical_crossentropy" if cfg["label_mode"] == "int"
            else "categorical_crossentropy")
    opt = keras.optimizers.Adam(learning_rate=cfg["learning_rate"])
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def get_callbacks(ckpt_dir, cfg):
    os.makedirs(ckpt_dir, exist_ok=True)
    return [
        keras.callbacks.ModelCheckpoint(os.path.join(ckpt_dir, "best.keras"),
            monitor="val_loss", save_best_only=True, verbose=1),
        keras.callbacks.ModelCheckpoint(os.path.join(ckpt_dir, "last.keras"),
            save_best_only=False, verbose=0),   # for resuming after disconnect
        keras.callbacks.EarlyStopping(monitor="val_loss",
            patience=cfg["early_stopping_patience"], restore_best_weights=True),
        keras.callbacks.CSVLogger(os.path.join(ckpt_dir, "history.csv"), append=True),
    ]

def save_config(cfg, path):
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2)

In [ ]:
import importlib, sys
import numpy as np
sys.path.insert(0, "/content/repo")
import src.train as T
importlib.reload(T)

cfg = dict(T.DEFAULT_CONFIG)
T.set_seed(cfg["seed"])
model = T.compile_model(
    T.build_cnn(cfg["input_shape"], cfg["num_classes"], cfg["filters"], cfg["dense_units"], cfg["dropout"]), cfg)
model.summary()

# Forward pass + shape checks, for several class counts
for C in [2, 4, 7, 22]:
    m = T.build_cnn(cfg["input_shape"], C)
    x = np.random.rand(8, *cfg["input_shape"]).astype("float32")
    y = m(x, training=False).numpy()
    assert y.shape == (8, C), y.shape
    assert np.allclose(y.sum(axis=1), 1.0, atol=1e-5)   # softmax sums to 1
    print(f"num_classes={C}: output {y.shape} OK")
print("SMOKE TEST PASSED (structure only, says nothing about accuracy)")

In [ ]:
DATA_DIR = f"{PROJECT_DRIVE}/data_processed"   # TODO: path from Hùng
CKPT_DIR = f"{PROJECT_DRIVE}/checkpoints/run_001"

def load_datasets(cfg):
    h, w, _ = cfg["input_shape"]
    kw = dict(image_size=(h, w), batch_size=cfg["batch_size"], seed=cfg["seed"],
              label_mode="int" if cfg["label_mode"] == "int" else "categorical")
    train = tf.keras.utils.image_dataset_from_directory(f"{DATA_DIR}/train", shuffle=True, **kw)
    val   = tf.keras.utils.image_dataset_from_directory(f"{DATA_DIR}/val", shuffle=False, **kw)
    norm = lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
    return (train.map(norm).prefetch(tf.data.AUTOTUNE),
            val.map(norm).prefetch(tf.data.AUTOTUNE))

def run_training(cfg, resume=False):
    T.set_seed(cfg["seed"])
    T.save_config(cfg, f"{CKPT_DIR}/config.json") if os.path.isdir(CKPT_DIR) else (os.makedirs(CKPT_DIR), T.save_config(cfg, f"{CKPT_DIR}/config.json"))
    train_ds, val_ds = load_datasets(cfg)
    initial_epoch = 0
    if resume and os.path.exists(f"{CKPT_DIR}/last.keras"):
        model = tf.keras.models.load_model(f"{CKPT_DIR}/last.keras")
        import pandas as pd
        initial_epoch = len(pd.read_csv(f"{CKPT_DIR}/history.csv"))
        print("Resuming from epoch", initial_epoch)
    else:
        model = T.compile_model(T.build_cnn(cfg["input_shape"], cfg["num_classes"],
                cfg["filters"], cfg["dense_units"], cfg["dropout"]), cfg)
    return model.fit(train_ds, validation_data=val_ds, epochs=cfg["epochs"],
                     initial_epoch=initial_epoch, callbacks=T.get_callbacks(CKPT_DIR, cfg))



In [ ]:
!printf "checkpoints/\n*.keras\n*.h5\ndata/\n" >> .gitignore
!git add src/train.py requirements.txt .gitignore
!git commit -m "feat(s1): CNN baseline, config, smoke test and training pipeline"
!git push -u origin feature/s1-cnn-baseline